In [ ]:

!pip install rasterio scikit-learn matplotlib joblib


In [ ]:
import os
import numpy as np
import rasterio
from glob import glob
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib
import matplotlib.pyplot as plt

def dice_coef(pred, true, smooth=1e-6):
    p = pred.flatten()
    t = true.flatten()
    inter = (p * t).sum()
    return (2 * inter + smooth) / (p.sum() + t.sum() + smooth)

def iou_score(pred, true, smooth=1e-6):
    p = pred.flatten()
    t = true.flatten()
    inter = (p * t).sum()
    union = p.sum() + t.sum() - inter
    return (inter + smooth) / (union + smooth)

def augment_tile(img, mask):
    aug_imgs = [img]
    aug_masks = [mask]
    aug_imgs.append(np.fliplr(img))
    aug_masks.append(np.fliplr(mask))
    return aug_imgs, aug_masks


In [ ]:
if __name__ == "__main__":
    DATA_DIR = '/kaggle/input/cloud-masking-dataset/content/train/data'
    MASK_DIR = '/kaggle/input/cloud-masking-dataset/content/train/masks'
    out_dir  = '/kaggle/working'

    os.makedirs(out_dir, exist_ok=True)

    img_paths = sorted(glob(os.path.join(DATA_DIR, '*.tif')))
    ids = [os.path.splitext(os.path.basename(p))[0] for p in img_paths]

    train_ids, val_ids = train_test_split(ids, test_size=0.2, random_state=42)
    print(f"Found {len(train_ids)} train images and {len(val_ids)} validation images.")

    X_list, y_list = [], []
    for img_id in train_ids:
        try:
            with rasterio.open(f'{DATA_DIR}/{img_id}.tif') as src:
                img = src.read([1,2,3,4]).astype(np.float32)
            with rasterio.open(f'{MASK_DIR}/{img_id}.tif') as src:
                mask = (src.read(1) > 0).astype(np.uint8)
        except Exception as e:
            print(f"Couldn't find{img_id}")
            continue

        aug_imgs, aug_masks = augment_tile(img, mask)
        for a_img, a_mask in zip(aug_imgs, aug_masks):
            feats = a_img.reshape(4, -1).T
            labs  = a_mask.flatten()
            N = feats.shape[0]
            sample_size = max(1, int(N * 0.005))
            idx = np.random.choice(N, sample_size, replace=False)
            X_list.append(feats[idx])
            y_list.append(labs[idx])


    X_train = np.vstack(X_list)
    y_train = np.hstack(y_list)
    print(f"Training on {X_train.shape[0]} pixels")

    rf = RandomForestClassifier(
        n_estimators=25,
        max_depth=20,
        n_jobs=-1,
        random_state=42,
        verbose=1
    )
    rf.fit(X_train, y_train)

    model_path = os.path.join(out_dir, 'random_forest.pkl')
    joblib.dump(rf, model_path)

    misclassified_threshold = 0.6
    max_to_display = 5
    shown = 0

    for img_id in val_ids:
        try:
            with rasterio.open(f'{DATA_DIR}/{img_id}.tif') as src:
                img = src.read([1,2,3,4]).astype(np.float32)
            with rasterio.open(f'{MASK_DIR}/{img_id}.tif') as src:
                mask_true = (src.read(1) > 0).astype(np.uint8)
        except Exception as e:
            print(f"Couldn't find {img_id}")
            continue

        H, W = mask_true.shape
        X_val = img.reshape(4, -1).T
        y_pred = rf.predict(X_val).reshape(H, W)

        d = dice_coef(y_pred, mask_true)
        j = iou_score(y_pred, mask_true)
        print(f"{img_id}: Dice={d:.4f}, IoU={j:.4f}")

        if j < misclassified_threshold and shown < max_to_display:
            fig, ax = plt.subplots(1, 3, figsize=(12,4))
            ax[0].imshow(np.transpose(img[:3], (1,2,0)) / 255.0)
            ax[0].set_title('RGB')
            ax[1].imshow(mask_true, cmap='gray')
            ax[1].set_title('GT Mask')
            ax[2].imshow(y_pred, cmap='gray')
            ax[2].set_title(f'RF Prediction (IoU={j:.2f})')
            plt.tight_layout()
            plt.show()
            shown += 1
